In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = '#cccccc'

# Theme colors matching the dataset's banner (night -> dawn gradient)
NIGHT, DUSK, DAWN, SUN, TEAL = '#1c1d42', '#5b4a76', '#d1875f', '#eba766', '#4fa3ad'

df = pd.read_csv('students.csv')
df.shape

(5, 6)

## New student form

Fill in the form below and click **Add student**. The address is converted to latitude/longitude, the distance to Rocky Mountain High School is calculated, and the record is appended to `students.csv`.

In [ ]:
%pip install -q ipywidgets requests

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import csv
import math
import requests

STUDENTS_FILE = 'students.csv'

# Rocky Mountain High School, 1300 W Swallow Rd, Fort Collins, CO 80526
SCHOOL_NAME = 'Rocky Mountain High School'
SCHOOL_LOCATION = (40.546241, -105.099610)  # (latitude, longitude)

GEOCODER_URL = 'https://geocoding.geo.census.gov/geocoder/locations/onelineaddress'


def geocode_address(address):
    """Return (latitude, longitude) for a US street address, or None if it can't be found."""
    params = {'address': address, 'benchmark': 'Public_AR_Current', 'format': 'json'}
    response = requests.get(GEOCODER_URL, params=params, timeout=20)
    response.raise_for_status()
    matches = response.json()['result']['addressMatches']
    if not matches:
        return None
    coords = matches[0]['coordinates']
    return (coords['y'], coords['x'])


def distance_miles(a, b):
    """Straight-line (great-circle) distance in miles between two (latitude, longitude) points."""
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 3958.8 * math.asin(math.sqrt(h))


def distance_from_school(address):
    """Return miles from the address to the school, or None if the address can't be found."""
    location = geocode_address(address)
    if location is None:
        return None
    return round(distance_miles(location, SCHOOL_LOCATION), 1)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

first_name = widgets.Text(description='First name')
last_name = widgets.Text(description='Last name')
email = widgets.Text(description='Email')
gender = widgets.Dropdown(description='Gender', options=['', 'Female', 'Male', 'Prefer not to say'])
grade = widgets.Dropdown(description='Grade', options=['', 9, 10, 11, 12])
street = widgets.Text(description='Street', placeholder='1300 W Swallow Rd')
city = widgets.Text(description='City', value='Fort Collins')
state = widgets.Text(description='State', value='CO')
zip_code = widgets.Text(description='ZIP')
submit = widgets.Button(description='Add student', button_style='primary')
message = widgets.Output()


def add_student(_):
    message.clear_output()
    with message:
        fields = [first_name, last_name, email, gender, grade, street, city, state, zip_code]
        missing = [w.description for w in fields if not str(w.value).strip()]
        if missing:
            print('Please fill in: ' + ', '.join(missing))
            return
        if '@' not in email.value or '.' not in email.value.split('@')[-1]:
            print('Please enter a valid email address.')
            return
        existing = pd.read_csv(STUDENTS_FILE, dtype=str).fillna('')
        taken = existing[existing['email'].str.strip().str.lower() == email.value.strip().lower()]
        if len(taken):
            other = taken.iloc[0]
            print(f'{email.value.strip()} is already used by #{other.student_id} {other.first_name} {other.last_name}.')
            return

        address = f'{street.value.strip()}, {city.value.strip()}, {state.value.strip()} {zip_code.value.strip()}'
        try:
            location = geocode_address(address)
        except requests.RequestException as e:
            print(f'Could not reach the address lookup service: {e}')
            return
        if location is None:
            print(f'Could not find "{address}". Please check the address and try again.')
            return

        latitude, longitude = round(location[0], 6), round(location[1], 6)
        distance = round(distance_miles(location, SCHOOL_LOCATION), 1)
        student_id = int(pd.read_csv(STUDENTS_FILE)['student_id'].max()) + 1
        with open(STUDENTS_FILE, 'a', newline='') as f:
            csv.writer(f).writerow([student_id, first_name.value.strip(), last_name.value.strip(),
                                    gender.value, grade.value, address, distance, latitude, longitude,
                                    email.value.strip()])
        print(f'Added #{student_id} {first_name.value.strip()} {last_name.value.strip()}: '
              f'{distance} miles from {SCHOOL_NAME}.')

        for w in (first_name, last_name, email, street, zip_code):
            w.value = ''
        gender.value = grade.value = ''


submit.on_click(add_student)
display(widgets.VBox([first_name, last_name, email, gender, grade, street, city, state, zip_code, submit, message]))

In [ ]:
pd.read_csv(STUDENTS_FILE)

## Student app

Opens the Students / Map / Friends app in its own window. Run this cell again to reopen the window.

In [ ]:
import app

app.launch()